# 03 · Generative: conditional VAE (Keras/TensorFlow) + Word2Vec sentiment condition
Buổi 14–15. The CVAE learns FashionMNIST product images conditioned on **category + a Word2Vec sentiment vector**.
Each training image gets a random sentiment mix and a matching brightness, so the decoder learns: positive → bright, negative → dark.
The trained decoder augments a dataset with sentiment-specific variants. Target: reconstruction loss (per-pixel MSE) < 0.01.

The API ships a decoder trained with this exact code; retraining here lets you reproduce or improve it. About 10 minutes on a T4.

In [ ]:
#@title Setup — run once per session (Runtime > Change runtime type > T4 GPU)
REPO_URL = "https://github.com/YoungsterDumb/reviewlens"  # @param {type:"string"}
HF_NAMESPACE = "DatDuc"  # @param {type:"string"}  your Hugging Face username
import os, sys, subprocess
if not os.path.exists("/content/reviewlens"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, "/content/reviewlens"], check=True)
else:
    subprocess.run(["git", "-C", "/content/reviewlens", "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/reviewlens/training/requirements.txt"], check=True)
sys.path.insert(0, "/content/reviewlens/training")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — switch the runtime to T4 GPU")

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # paste a token with WRITE access (huggingface.co/settings/tokens)
METRICS_REPO = f"{HF_NAMESPACE}/reviewlens-metrics"

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"  # Keras 3 on TensorFlow for training
from pathlib import Path
import numpy as np
from reviewlens_train.data import load_fashion_mnist, split_arrays, load_absa
from reviewlens_train.cvae_train import train_word2vec, sentiment_anchors, train_cvae, evaluate_cvae, export
x, y = load_fashion_mnist(Path("/content/data/fashion"))
splits = split_arrays(len(x))
print(x.shape, [len(s) for s in splits])

## Word2Vec on the review corpus (EN + VI) → sentiment anchors

In [ ]:
absa = load_absa(Path("/content/data"))
corpus = [r["text"] for part in absa.values() for r in part]
w2v = train_word2vec(corpus)
anchors, used = sentiment_anchors(w2v)
print(used)
print("cos(pos, neg) =", float(anchors["positive"] @ anchors["negative"]))

In [ ]:
cvae, encoder, decoder, history = train_cvae(x, y, splits, anchors, epochs=30, batch_size=256)
metrics = evaluate_cvae(encoder, decoder, x, y, splits[2], anchors)
metrics

In [ ]:
import matplotlib.pyplot as plt
from app.core.cvae import condition_vector, CATEGORIES
z = np.random.default_rng(0).standard_normal((1, 16)).astype("float32")
fig, axes = plt.subplots(3, 7, figsize=(10, 4.6))
for row, cat in enumerate([7, 8, 0]):
    for col in range(7):
        t = col / 6
        c = condition_vector(cat, {"negative": 1 - t, "neutral": 0, "positive": t}, anchors)[None]
        axes[row, col].imshow(np.asarray(decoder([z, c]))[0, ..., 0], cmap="gray", vmin=0, vmax=1)
        axes[row, col].axis("off")
    axes[row, 0].set_title(CATEGORIES[cat], loc="left", fontsize=9)
fig.suptitle("Same latent vector, sentiment from negative (left) to positive (right)")
plt.show()

In [ ]:
from reviewlens_train.hub import push_folder, update_metrics
out = export(decoder, anchors, used, metrics, Path("/content/out/cvae"))
push_folder(out, f"{HF_NAMESPACE}/reviewlens-cvae", "CVAE decoder + sentiment anchors")
update_metrics(METRICS_REPO, {"vae_recon_mse": metrics["vae_recon_mse"], "vae_kl": metrics["vae_kl"]},
               extra={"cvae_repo": f"{HF_NAMESPACE}/reviewlens-cvae"})